# 03 - LightGCN Baseline

Train and evaluate a compact LightGCN collaborative-filtering baseline on the
MovieLens-1M cold-start protocol produced by notebook 02. This notebook uses
interactions only: no title, genre, user demographic, or item-count features.

## Notebook Linkage and Work Plan

**Input from notebook 02:** the `ml1m-coldstart-v1` protocol manifest plus
verified `tuning_train`, `final_train`, `validation_tasks`, `evaluation_tasks`,
`users`, and `items` CSV artifacts. Those files are the dataset interface for
this baseline, so no separate dataset notebook is needed here.

**What this notebook does:**
1. Verify and load the protocol handoff.
2. Define a minimal LightGCN model, BPR training loop, phase graph builder, and
   row-level CTR metrics.
3. Tune only on `tuning_train` plus validation support rows, selecting one
   configuration and one F1 threshold per phase from validation query labels.
4. Refit the selected configuration on `final_train`, evaluate Cold/Warm A/B/C
   on new items, and apply the frozen validation thresholds.
5. Publish a compact LightGCN result bundle for notebooks 06-08.

## Verified Protocol Load

Locate notebook 02 locally or from a mounted Kaggle input, verify the pointer
manifest, immutable generation manifest, artifact hashes, row counts, columns,
and declared dtypes, then load the exact tables. This preserves the leakage
boundary: LightGCN sees only protocol-approved rows and indexed identifiers.

In [1]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import platform
import random
import shutil
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

REQUIRED_PACKAGES = ["numpy", "pandas", "torch", "IPython"]
MISSING_PACKAGES = [
    package for package in REQUIRED_PACKAGES if importlib.util.find_spec(package) is None
]
if MISSING_PACKAGES:
    raise RuntimeError(
        "Notebook 03 requires these packages in the active kernel: "
        + ", ".join(MISSING_PACKAGES)
        + ". Run it in the project ML/Kaggle environment used for model notebooks."
    )

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from torch import nn


def show_records(records: Iterable[dict[str, Any]]) -> None:
    display(pd.DataFrame(list(records)))


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root: Path, relative_path: str) -> Path:
    resolved_root = root.resolve()
    resolved = (resolved_root / relative_path).resolve()
    resolved.relative_to(resolved_root)
    return resolved


def project_root(start: Path) -> Path:
    override = os.environ.get("COLDSTART_PROJECT_ROOT")
    if override:
        return Path(override).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate.resolve()
    return start.resolve()


EXECUTION_CONTEXT = "kaggle" if Path("/kaggle/input").exists() else "local"
PROJECT_ROOT = project_root(Path.cwd())
WORKSPACE_ROOT = Path(
    os.environ.get(
        "COLDSTART_WORKSPACE_ROOT",
        "/kaggle/working" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / ".notebook",
    )
).expanduser().resolve()
INPUT_ROOT = Path(
    os.environ.get(
        "COLDSTART_INPUT_ROOT",
        "/kaggle/input" if EXECUTION_CONTEXT == "kaggle" else PROJECT_ROOT / "data",
    )
).expanduser().resolve()
ARTIFACT_ROOT = Path(
    os.environ.get("COLDSTART_ARTIFACT_ROOT", WORKSPACE_ROOT / "artifacts")
).expanduser().resolve()
PROTOCOL_RELATIVE_MANIFEST = Path("protocols/ml-1m/coldstart-v1/manifest.json")
MODEL_OUTPUT_ROOT = ARTIFACT_ROOT / "models" / "ml-1m" / "lightgcn-v1"
FAST_DEV_RUN = os.environ.get("COLDSTART_FAST_DEV_RUN", "0") == "1"

requested_device = os.environ.get("COLDSTART_DEVICE")
if requested_device:
    DEVICE = torch.device(requested_device)
    if DEVICE.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("COLDSTART_DEVICE requests CUDA, but torch.cuda is unavailable")
else:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = int(os.environ.get("COLDSTART_SEED", "2025"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

base_epochs = int(os.environ.get("COLDSTART_LIGHTGCN_EPOCHS", "12"))
base_sample_size = int(os.environ.get("COLDSTART_LIGHTGCN_SAMPLE_SIZE", "65536"))
if FAST_DEV_RUN:
    base_epochs = min(base_epochs, 2)
    base_sample_size = min(base_sample_size, 8192)

RUN_CONFIG: dict[str, Any] = {
    "schema_version": "lightgcn-baseline-v1",
    "seed": SEED,
    "fast_dev_run": FAST_DEV_RUN,
    "candidate_configs": [
        {
            "embedding_dim": 64,
            "n_layers": 2,
            "learning_rate": 0.003,
            "weight_decay": 1e-4,
            "epochs": base_epochs,
            "sample_size": base_sample_size,
        },
        {
            "embedding_dim": 64,
            "n_layers": 3,
            "learning_rate": 0.003,
            "weight_decay": 1e-4,
            "epochs": base_epochs,
            "sample_size": base_sample_size,
        },
    ][:1 if FAST_DEV_RUN else 2],
    "phase_order": ["Cold", "Warm A", "Warm B", "Warm C"],
    "scoring_batch_size": int(os.environ.get("COLDSTART_LIGHTGCN_SCORE_BATCH", "131072")),
    "training_feedback": "all visible interactions are implicit LightGCN edges/BPR positives",
    "negative_item_pool": "items visible in the current training base only",
}
RUN_CONFIG_HASH = hashlib.sha256(
    json.dumps(RUN_CONFIG, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()

show_records(
    [
        {
            "execution_context": EXECUTION_CONTEXT,
            "python": platform.python_version(),
            "torch": torch.__version__,
            "device": str(DEVICE),
            "artifact_root": str(ARTIFACT_ROOT),
            "model_output_root": str(MODEL_OUTPUT_ROOT),
            "run_config_sha256": RUN_CONFIG_HASH,
        }
    ]
)


def protocol_candidates() -> list[tuple[Path, Path]]:
    candidates: list[tuple[Path, Path]] = []
    explicit = os.environ.get("COLDSTART_PROTOCOL_ROOT")
    roots = [Path(explicit).expanduser()] if explicit else []
    roots.extend([PROJECT_ROOT / ".notebook" / "artifacts", ARTIFACT_ROOT])

    for root in roots:
        pointer = root / PROTOCOL_RELATIVE_MANIFEST
        if pointer.is_file():
            candidates.append((root.resolve(), pointer.resolve()))
        direct = root / "manifest.json"
        if root.name == "coldstart-v1" and direct.is_file():
            candidates.append((root.parents[2].resolve(), direct.resolve()))

    if INPUT_ROOT.is_dir():
        for pointer in sorted(INPUT_ROOT.rglob("manifest.json")):
            parent = pointer.parent
            if (
                parent.name == "coldstart-v1"
                and parent.parent.name == "ml-1m"
                and parent.parent.parent.name == "protocols"
            ):
                candidates.append((pointer.parents[3].resolve(), pointer.resolve()))

    unique: list[tuple[Path, Path]] = []
    seen: set[str] = set()
    for root, pointer in candidates:
        key = str(pointer)
        if key not in seen:
            seen.add(key)
            unique.append((root, pointer))
    return unique


def load_verified_protocol(root: Path, pointer: Path) -> tuple[dict[str, Any], dict[str, pd.DataFrame]]:
    pointer_bytes = pointer.read_bytes()
    manifest = json.loads(pointer_bytes)
    if manifest.get("protocol_schema_version") != "ml1m-coldstart-v1":
        raise ValueError(f"Unexpected protocol schema: {manifest.get('protocol_schema_version')!r}")
    if manifest.get("protocol_status") != "PASS":
        raise ValueError(f"Protocol status is not PASS: {manifest.get('protocol_status')!r}")
    if not all(row.get("status") == "PASS" for row in manifest.get("checks", [])):
        raise ValueError("One or more notebook-02 protocol checks did not pass")

    bundle_manifest = resolve_inside(root, manifest["bundle_manifest"])
    if bundle_manifest.read_bytes() != pointer_bytes:
        raise ValueError("Protocol pointer and immutable generation manifest differ")

    tables: dict[str, pd.DataFrame] = {}
    for name, artifact in manifest["artifacts"].items():
        path = resolve_inside(root, artifact["path"])
        if not path.is_file() or sha256_file(path) != artifact["sha256"]:
            raise ValueError(f"Protocol artifact verification failed: {name}")
        schema = manifest["output_schemas"][name]
        table = pd.read_csv(path, dtype=schema["read_csv_dtypes"])
        if list(table.columns) != schema["columns"] or len(table) != artifact["rows"]:
            raise ValueError(f"Protocol table contract failed: {name}")
        tables[name] = table
    return manifest, tables


PROTOCOL_ERRORS: list[str] = []
PROTOCOL_ROOT = None
PROTOCOL_POINTER = None
PROTOCOL_MANIFEST = None
TABLES = None
for candidate_root, candidate_pointer in protocol_candidates():
    try:
        PROTOCOL_MANIFEST, TABLES = load_verified_protocol(candidate_root, candidate_pointer)
        PROTOCOL_ROOT, PROTOCOL_POINTER = candidate_root, candidate_pointer
        break
    except Exception as error:
        PROTOCOL_ERRORS.append(f"{candidate_pointer}: {error}")

if PROTOCOL_MANIFEST is None or TABLES is None or PROTOCOL_POINTER is None:
    raise RuntimeError(
        "No valid notebook-02 protocol bundle found. Set COLDSTART_PROTOCOL_ROOT. "
        + " | ".join(PROTOCOL_ERRORS)
    )

TUNING_TRAIN = TABLES["tuning_train"]
FINAL_TRAIN = TABLES["final_train"]
VALIDATION_TASKS = TABLES["validation_tasks"]
EVALUATION_TASKS = TABLES["evaluation_tasks"]
USERS = TABLES["users"]
ITEMS = TABLES["items"]
N_USERS = int(USERS["user_idx"].max()) + 1
N_ITEMS = int(ITEMS["item_idx"].max()) + 1
PHASES = tuple(RUN_CONFIG["phase_order"])
PHASE_SUPPORT_ROLES = {
    "Cold": [],
    "Warm A": ["warm_a"],
    "Warm B": ["warm_a", "warm_b"],
    "Warm C": ["warm_a", "warm_b", "warm_c"],
}
PROTOCOL_POINTER_SHA256 = sha256_file(PROTOCOL_POINTER)

show_records(
    [
        {
            "protocol_bundle": PROTOCOL_MANIFEST["bundle_id"],
            "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
            "users": N_USERS,
            "items": N_ITEMS,
            "tuning_train_rows": len(TUNING_TRAIN),
            "final_train_rows": len(FINAL_TRAIN),
            "validation_query_rows": int(VALIDATION_TASKS["role"].eq("query").sum()),
            "evaluation_query_rows": int(EVALUATION_TASKS["role"].eq("query").sum()),
        }
    ]
)

,execution_context,python,torch,device,artifact_root,model_output_root,run_config_sha256
0,kaggle,3.12.13,2.10.0+cu128,cuda,/kaggle/working/artifacts,/kaggle/working/artifacts/models/ml-1m/lightgc...,9281074b23df448710be1b8c39e2b6ef17a49c41162110...


,protocol_bundle,protocol_pointer_sha256,users,items,tuning_train_rows,final_train_rows,validation_query_rows,evaluation_query_rows
0,20260716T000035-d094523cf74d,f6461f73814475752e5db6bf6b5ae373ca92e3c049aff8...,6040,2375,684728,854530,152762,57569


## LightGCN, Phase Graphs, and Metrics

LightGCN keeps only ID embeddings and normalized user-item propagation. The
base graph is `tuning_train` during selection and `final_train` after freezing
choices. Warm phases add support rows cumulatively, while Cold leaves new items
isolated instead of dropping them. F1 thresholds are fitted only on validation
query rows; evaluation uses those frozen thresholds unchanged.

In [2]:
EDGE_COLUMNS = ["user_idx", "item_idx"]


def edge_array(table: pd.DataFrame) -> np.ndarray:
    return table[EDGE_COLUMNS].to_numpy(dtype=np.int64, copy=True)


def phase_edges(base: pd.DataFrame, tasks: pd.DataFrame, phase: str) -> np.ndarray:
    roles = PHASE_SUPPORT_ROLES[phase]
    if not roles:
        return edge_array(base)
    support = tasks.loc[tasks["role"].isin(roles), EDGE_COLUMNS]
    return pd.concat([base[EDGE_COLUMNS], support], ignore_index=True).to_numpy(
        dtype=np.int64, copy=True
    )


def build_norm_adj(edges: np.ndarray) -> torch.Tensor:
    users = torch.as_tensor(edges[:, 0], dtype=torch.long, device=DEVICE)
    items = torch.as_tensor(edges[:, 1] + N_USERS, dtype=torch.long, device=DEVICE)
    row = torch.cat([users, items])
    col = torch.cat([items, users])
    degree = torch.bincount(row, minlength=N_USERS + N_ITEMS).float()
    degree = degree.clamp_min_(1.0)
    values = torch.rsqrt(degree[row] * degree[col])
    indices = torch.stack([row, col])
    return torch.sparse_coo_tensor(
        indices, values, (N_USERS + N_ITEMS, N_USERS + N_ITEMS), device=DEVICE
    ).coalesce()


class LightGCN(nn.Module):
    def __init__(self, n_users: int, n_items: int, embedding_dim: int, n_layers: int):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.n_layers = n_layers
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def propagate(self, norm_adj: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        embeddings = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)
        layers = [embeddings]
        for _ in range(self.n_layers):
            embeddings = torch.sparse.mm(norm_adj, embeddings)
            layers.append(embeddings)
        output = torch.stack(layers, dim=0).mean(dim=0)
        return output[: self.n_users], output[self.n_users :]


def make_seen_sets(edges: np.ndarray) -> list[set[int]]:
    seen = [set() for _ in range(N_USERS)]
    for user_idx, item_idx in edges:
        seen[int(user_idx)].add(int(item_idx))
    return seen


def sample_negative_items(
    users: np.ndarray,
    item_pool: np.ndarray,
    seen_by_user: list[set[int]],
    rng: np.random.Generator,
) -> np.ndarray:
    negatives = rng.choice(item_pool, size=len(users), replace=True).astype(np.int64)
    bad = np.zeros(len(users), dtype=bool)
    for _ in range(20):
        bad = np.fromiter(
            (int(item) in seen_by_user[int(user)] for user, item in zip(users, negatives)),
            dtype=bool,
            count=len(users),
        )
        if not bad.any():
            return negatives
        negatives[bad] = rng.choice(item_pool, size=int(bad.sum()), replace=True)

    for idx in np.flatnonzero(bad):
        seen = seen_by_user[int(users[idx])]
        available = np.array([item for item in item_pool if int(item) not in seen], dtype=np.int64)
        negatives[idx] = rng.choice(available if len(available) else item_pool)
    return negatives


def train_lightgcn(
    train_edges: np.ndarray,
    config: dict[str, Any],
    seed: int,
    run_name: str,
) -> tuple[LightGCN, pd.DataFrame, float]:
    start = time.perf_counter()
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    item_pool = np.unique(train_edges[:, 1]).astype(np.int64)
    seen_by_user = make_seen_sets(train_edges)
    norm_adj = build_norm_adj(train_edges)
    model = LightGCN(
        N_USERS,
        N_ITEMS,
        embedding_dim=int(config["embedding_dim"]),
        n_layers=int(config["n_layers"]),
    ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=float(config["learning_rate"]))
    history: list[dict[str, Any]] = []
    sample_size = int(config["sample_size"])

    for epoch in range(1, int(config["epochs"]) + 1):
        model.train()
        sampled = rng.integers(0, len(train_edges), size=sample_size)
        users_np = train_edges[sampled, 0]
        positives_np = train_edges[sampled, 1]
        negatives_np = sample_negative_items(users_np, item_pool, seen_by_user, rng)

        users = torch.as_tensor(users_np, dtype=torch.long, device=DEVICE)
        positives = torch.as_tensor(positives_np, dtype=torch.long, device=DEVICE)
        negatives = torch.as_tensor(negatives_np, dtype=torch.long, device=DEVICE)

        optimizer.zero_grad(set_to_none=True)
        user_embeddings, item_embeddings = model.propagate(norm_adj)
        pos_scores = (user_embeddings[users] * item_embeddings[positives]).sum(dim=1)
        neg_scores = (user_embeddings[users] * item_embeddings[negatives]).sum(dim=1)
        bpr_loss = -F.logsigmoid(pos_scores - neg_scores).mean()
        regularizer = (
            model.user_embedding(users).pow(2).sum()
            + model.item_embedding(positives).pow(2).sum()
            + model.item_embedding(negatives).pow(2).sum()
        ) / (2.0 * len(users_np))
        loss = bpr_loss + float(config["weight_decay"]) * regularizer
        loss.backward()
        optimizer.step()

        history.append(
            {
                "run_name": run_name,
                "epoch": epoch,
                "loss": float(loss.detach().cpu()),
                "bpr_loss": float(bpr_loss.detach().cpu()),
                "regularizer": float(regularizer.detach().cpu()),
                "sample_size": sample_size,
            }
        )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return model, pd.DataFrame(history), time.perf_counter() - start


def score_query_rows(model: LightGCN, graph_edges: np.ndarray, query_rows: pd.DataFrame, phase: str) -> pd.DataFrame:
    model.eval()
    norm_adj = build_norm_adj(graph_edges)
    scores: list[np.ndarray] = []
    batch_size = int(RUN_CONFIG["scoring_batch_size"])
    with torch.no_grad():
        user_embeddings, item_embeddings = model.propagate(norm_adj)
        users_np = query_rows["user_idx"].to_numpy(dtype=np.int64)
        items_np = query_rows["item_idx"].to_numpy(dtype=np.int64)
        for start in range(0, len(query_rows), batch_size):
            end = min(start + batch_size, len(query_rows))
            users = torch.as_tensor(users_np[start:end], dtype=torch.long, device=DEVICE)
            items = torch.as_tensor(items_np[start:end], dtype=torch.long, device=DEVICE)
            batch_scores = (user_embeddings[users] * item_embeddings[items]).sum(dim=1)
            scores.append(batch_scores.detach().cpu().numpy())
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    result = query_rows[
        ["source_row", "user_id", "user_idx", "item_id", "item_idx", "label"]
    ].copy()
    result.insert(0, "phase", phase)
    result["score"] = np.concatenate(scores).astype(np.float32)
    return result


def roc_auc(labels: np.ndarray, scores: np.ndarray) -> float:
    labels = labels.astype(np.int64)
    positives = int(labels.sum())
    negatives = int(len(labels) - positives)
    if positives == 0 or negatives == 0:
        return float("nan")

    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    ranks = np.arange(1, len(scores) + 1, dtype=np.float64)
    start = 0
    while start < len(scores):
        end = start + 1
        while end < len(scores) and sorted_scores[end] == sorted_scores[start]:
            end += 1
        if end - start > 1:
            ranks[start:end] = ranks[start:end].mean()
        start = end
    original_ranks = np.empty_like(ranks)
    original_ranks[order] = ranks
    return float((original_ranks[labels == 1].sum() - positives * (positives + 1) / 2) / (positives * negatives))


def best_f1_threshold(labels: np.ndarray, scores: np.ndarray) -> tuple[float, float]:
    labels = labels.astype(np.int64)
    order = np.argsort(-scores, kind="mergesort")
    sorted_labels = labels[order]
    sorted_scores = scores[order]
    tp = np.cumsum(sorted_labels)
    fp = np.cumsum(1 - sorted_labels)
    fn = int(sorted_labels.sum()) - tp
    denominator = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, denominator, out=np.zeros_like(tp, dtype=np.float64), where=denominator > 0)
    best = int(np.argmax(f1))
    return float(sorted_scores[best]), float(f1[best])


def binary_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, Any]:
    labels = labels.astype(np.int64)
    predictions = scores >= threshold
    tp = int(((predictions == 1) & (labels == 1)).sum())
    fp = int(((predictions == 1) & (labels == 0)).sum())
    fn = int(((predictions == 0) & (labels == 1)).sum())
    tn = int(((predictions == 0) & (labels == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "rows": int(len(labels)),
        "positives": int(labels.sum()),
        "threshold": float(threshold),
        "accuracy": float((tp + tn) / len(labels)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": roc_auc(labels, scores),
        "predicted_positive_rate": float(predictions.mean()),
        "score_mean": float(scores.mean()),
        "score_std": float(scores.std()),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


def summarize_predictions(
    predictions: pd.DataFrame,
    split: str,
    thresholds: dict[str, float] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    threshold_rows: list[dict[str, Any]] = []
    metric_rows: list[dict[str, Any]] = []
    for phase in PHASES:
        phase_predictions = predictions[predictions["phase"].eq(phase)]
        labels = phase_predictions["label"].to_numpy(dtype=np.int64)
        scores = phase_predictions["score"].to_numpy(dtype=np.float64)
        if thresholds is None:
            threshold, best_f1 = best_f1_threshold(labels, scores)
        else:
            threshold, best_f1 = float(thresholds[phase]), np.nan
        threshold_rows.append(
            {"split": split, "phase": phase, "threshold": threshold, "validation_best_f1": best_f1}
        )
        metric_rows.append(
            {"split": split, "phase": phase, **binary_metrics(labels, scores, threshold)}
        )
    return pd.DataFrame(threshold_rows), pd.DataFrame(metric_rows)

## Tune on Validation Only

Each candidate trains on `tuning_train`. Validation phase graphs add only the
matching old-validation support rows, and thresholds come only from validation
query labels. The chosen configuration is the one with the best mean phase F1;
final new-item labels are not touched in this cell.

In [3]:
TUNING_EDGES = edge_array(TUNING_TRAIN)
VALIDATION_QUERY = VALIDATION_TASKS[VALIDATION_TASKS["role"].eq("query")].reset_index(drop=True)
TRAINING_HISTORY_PARTS: list[pd.DataFrame] = []
VALIDATION_METRIC_PARTS: list[pd.DataFrame] = []
VALIDATION_THRESHOLD_PARTS: list[pd.DataFrame] = []
CONFIG_SUMMARY_ROWS: list[dict[str, Any]] = []
selected_payload: dict[str, Any] | None = None

for config_index, config in enumerate(RUN_CONFIG["candidate_configs"], start=1):
    config_name = f"cfg{config_index:02d}_L{config['n_layers']}_D{config['embedding_dim']}"
    model, history, training_seconds = train_lightgcn(
        TUNING_EDGES, config, seed=SEED + config_index, run_name=config_name
    )
    history["config_name"] = config_name
    TRAINING_HISTORY_PARTS.append(history)

    validation_prediction_parts = []
    for phase in PHASES:
        validation_prediction_parts.append(
            score_query_rows(
                model,
                phase_edges(TUNING_TRAIN, VALIDATION_TASKS, phase),
                VALIDATION_QUERY,
                phase,
            )
        )
    validation_predictions = pd.concat(validation_prediction_parts, ignore_index=True)
    thresholds, metrics = summarize_predictions(validation_predictions, split="validation")
    thresholds["config_name"] = config_name
    metrics["config_name"] = config_name
    VALIDATION_THRESHOLD_PARTS.append(thresholds)
    VALIDATION_METRIC_PARTS.append(metrics)

    mean_f1 = float(metrics["f1"].mean())
    mean_auc = float(metrics["roc_auc"].mean())
    summary = {
        "config_name": config_name,
        "mean_validation_f1": mean_f1,
        "mean_validation_auc": mean_auc,
        "training_seconds": float(training_seconds),
        **config,
    }
    CONFIG_SUMMARY_ROWS.append(summary)
    candidate_payload = {
        "config_name": config_name,
        "config": dict(config),
        "summary": summary,
        "thresholds": thresholds,
        "metrics": metrics,
        "predictions": validation_predictions,
    }
    if selected_payload is None or (mean_f1, mean_auc) > (
        selected_payload["summary"]["mean_validation_f1"],
        selected_payload["summary"]["mean_validation_auc"],
    ):
        selected_payload = candidate_payload

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

if selected_payload is None:
    raise RuntimeError("No LightGCN candidate was trained")

TRAINING_HISTORY = pd.concat(TRAINING_HISTORY_PARTS, ignore_index=True)
VALIDATION_METRICS_ALL = pd.concat(VALIDATION_METRIC_PARTS, ignore_index=True)
VALIDATION_THRESHOLDS_ALL = pd.concat(VALIDATION_THRESHOLD_PARTS, ignore_index=True)
CONFIG_SUMMARY = pd.DataFrame(CONFIG_SUMMARY_ROWS).sort_values(
    ["mean_validation_f1", "mean_validation_auc"], ascending=False
)
SELECTED_CONFIG_NAME = str(selected_payload["config_name"])
SELECTED_CONFIG = dict(selected_payload["config"])
SELECTED_SUMMARY = dict(selected_payload["summary"])
VALIDATION_PREDICTIONS = selected_payload["predictions"].copy()
VALIDATION_THRESHOLDS = selected_payload["thresholds"].copy()
VALIDATION_METRICS = selected_payload["metrics"].copy()
THRESHOLD_BY_PHASE = dict(zip(VALIDATION_THRESHOLDS["phase"], VALIDATION_THRESHOLDS["threshold"]))

display(CONFIG_SUMMARY)
display(VALIDATION_METRICS[["config_name", "phase", "rows", "positives", "threshold", "f1", "roc_auc"]])
show_records([{**SELECTED_SUMMARY, "selected_config_name": SELECTED_CONFIG_NAME}])

,config_name,mean_validation_f1,mean_validation_auc,training_seconds,embedding_dim,n_layers,learning_rate,weight_decay,epochs,sample_size
0,cfg01_L2_D64,0.759379,0.496580,13.095169,64,2,0.003,0.0001,12,65536
1,cfg02_L3_D64,0.759375,0.496464,7.292285,64,3,0.003,0.0001,12,65536


,config_name,phase,rows,positives,threshold,f1,roc_auc
0,cfg01_L2_D64,Cold,152762,93504,-0.041051,0.759377,0.495550
1,cfg01_L2_D64,Warm A,152762,93504,-0.037596,0.759380,0.497207
2,cfg01_L2_D64,Warm B,152762,93504,-0.037102,0.759380,0.496694
3,cfg01_L2_D64,Warm C,152762,93504,-0.037513,0.759380,0.496870


,config_name,mean_validation_f1,mean_validation_auc,training_seconds,embedding_dim,n_layers,learning_rate,weight_decay,epochs,sample_size,selected_config_name
0,cfg01_L2_D64,0.759379,0.49658,13.095169,64,2,0.003,0.0001,12,65536,cfg01_L2_D64


## Final Refit, New-Item Evaluation, and Export

Refit one LightGCN from scratch on `final_train`, then score the new-item query
rows under Cold/Warm A/B/C. The thresholds remain the validation thresholds
chosen above. The published bundle contains enough for result notebooks: config,
checkpoint, histories, thresholds, predictions, metrics, diagnostics, and a
manifest with hashes.

In [4]:
FINAL_EDGES = edge_array(FINAL_TRAIN)
EVALUATION_QUERY = EVALUATION_TASKS[EVALUATION_TASKS["role"].eq("query")].reset_index(drop=True)
FINAL_MODEL, FINAL_HISTORY, FINAL_TRAINING_SECONDS = train_lightgcn(
    FINAL_EDGES,
    SELECTED_CONFIG,
    seed=SEED + 10_000,
    run_name=f"final_refit_{SELECTED_CONFIG_NAME}",
)
FINAL_HISTORY["config_name"] = SELECTED_CONFIG_NAME
TRAINING_HISTORY = pd.concat([TRAINING_HISTORY, FINAL_HISTORY], ignore_index=True)

evaluation_prediction_parts = []
for phase in PHASES:
    evaluation_prediction_parts.append(
        score_query_rows(
            FINAL_MODEL,
            phase_edges(FINAL_TRAIN, EVALUATION_TASKS, phase),
            EVALUATION_QUERY,
            phase,
        )
    )
EVALUATION_PREDICTIONS = pd.concat(evaluation_prediction_parts, ignore_index=True)
_, EVALUATION_METRICS = summarize_predictions(
    EVALUATION_PREDICTIONS, split="evaluation", thresholds=THRESHOLD_BY_PHASE
)
EVALUATION_METRICS["config_name"] = SELECTED_CONFIG_NAME

DIAGNOSTICS = pd.DataFrame(
    [
        {
            "phase": phase,
            "base_train_edges": len(FINAL_TRAIN),
            "support_edges": int(EVALUATION_TASKS["role"].isin(PHASE_SUPPORT_ROLES[phase]).sum()),
            "graph_edges": len(phase_edges(FINAL_TRAIN, EVALUATION_TASKS, phase)),
            "query_rows": int(EVALUATION_QUERY.shape[0]),
            "query_users": int(EVALUATION_QUERY["user_idx"].nunique()),
            "query_items": int(EVALUATION_QUERY["item_idx"].nunique()),
            "threshold": float(THRESHOLD_BY_PHASE[phase]),
        }
        for phase in PHASES
    ]
)
DIAGNOSTICS["final_training_seconds"] = float(FINAL_TRAINING_SECONDS)
DIAGNOSTICS["device"] = str(DEVICE)

PRE_EXPORT_CHECKS: list[dict[str, Any]] = []


def pre_export_check(name: str, condition: bool, observed: Any, expected: Any) -> None:
    PRE_EXPORT_CHECKS.append(
        {"check": name, "status": "PASS" if condition else "FAIL", "observed": observed, "expected": expected}
    )


expected_validation_predictions = int(VALIDATION_QUERY.shape[0] * len(PHASES))
expected_evaluation_predictions = int(EVALUATION_QUERY.shape[0] * len(PHASES))
final_excludes_evaluation_items = set(FINAL_TRAIN["item_idx"]).isdisjoint(
    set(EVALUATION_TASKS["item_idx"])
)

pre_export_check("protocol schema", PROTOCOL_MANIFEST["protocol_schema_version"] == "ml1m-coldstart-v1", PROTOCOL_MANIFEST["protocol_schema_version"], "ml1m-coldstart-v1")
pre_export_check("phase thresholds complete", set(THRESHOLD_BY_PHASE) == set(PHASES), sorted(THRESHOLD_BY_PHASE), sorted(PHASES))
pre_export_check("validation predictions complete", len(VALIDATION_PREDICTIONS) == expected_validation_predictions, len(VALIDATION_PREDICTIONS), expected_validation_predictions)
pre_export_check("evaluation predictions complete", len(EVALUATION_PREDICTIONS) == expected_evaluation_predictions, len(EVALUATION_PREDICTIONS), expected_evaluation_predictions)
pre_export_check("evaluation phases complete", set(EVALUATION_METRICS["phase"]) == set(PHASES), sorted(EVALUATION_METRICS["phase"]), sorted(PHASES))
pre_export_check("final train excludes evaluation items", final_excludes_evaluation_items, final_excludes_evaluation_items, True)

if not all(row["status"] == "PASS" for row in PRE_EXPORT_CHECKS):
    display(pd.DataFrame(PRE_EXPORT_CHECKS))
    raise RuntimeError("LightGCN semantic checks failed; artifacts were not published")


def json_ready(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    temporary.write_text(content, encoding="utf-8")
    temporary.replace(path)


def write_csv(path: Path, table: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    table.to_csv(temporary, index=False, lineterminator="\n")
    temporary.replace(path)


def relative_output(path: Path) -> str:
    return str(path.resolve().relative_to(ARTIFACT_ROOT.resolve()))


def csv_schema(table: pd.DataFrame) -> dict[str, Any]:
    def dtype_name(dtype: Any) -> str:
        name = str(dtype)
        return "string" if name in {"str", "string"} or name.startswith("string") else name

    return {
        "columns": list(table.columns),
        "read_csv_dtypes": {column: dtype_name(dtype) for column, dtype in table.dtypes.items()},
    }


bundle_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:12]
staging_root = MODEL_OUTPUT_ROOT / f".staging-{bundle_id}"
generation_root = MODEL_OUTPUT_ROOT / "generations" / bundle_id
pointer_path = MODEL_OUTPUT_ROOT / "manifest.json"
staging_root.mkdir(parents=True, exist_ok=False)

OUTPUT_TABLES = {
    "training_history": TRAINING_HISTORY,
    "config_summary": CONFIG_SUMMARY,
    "validation_thresholds": VALIDATION_THRESHOLDS,
    "validation_metrics": VALIDATION_METRICS,
    "validation_predictions": VALIDATION_PREDICTIONS,
    "evaluation_metrics": EVALUATION_METRICS,
    "evaluation_predictions": EVALUATION_PREDICTIONS,
    "diagnostics": DIAGNOSTICS,
}
OUTPUT_ARTIFACTS: dict[str, Any] = {}

selected_config_payload = {
    "schema_version": RUN_CONFIG["schema_version"],
    "selected_config_name": SELECTED_CONFIG_NAME,
    "selected_config": SELECTED_CONFIG,
    "selected_summary": SELECTED_SUMMARY,
    "threshold_by_phase": THRESHOLD_BY_PHASE,
    "run_config_sha256": RUN_CONFIG_HASH,
    "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
}

try:
    for name, table in OUTPUT_TABLES.items():
        staging_path = staging_root / f"{name}.csv"
        published_path = generation_root / f"{name}.csv"
        write_csv(staging_path, table)
        OUTPUT_ARTIFACTS[name] = {
            "path": relative_output(published_path),
            "sha256": sha256_file(staging_path),
            "rows": len(table),
        }

    selected_config_path = staging_root / "selected_config.json"
    write_text(
        selected_config_path,
        json.dumps(json_ready(selected_config_payload), indent=2, sort_keys=True) + "\n",
    )
    OUTPUT_ARTIFACTS["selected_config"] = {
        "path": relative_output(generation_root / "selected_config.json"),
        "sha256": sha256_file(selected_config_path),
    }

    checkpoint_path = staging_root / "final_model.pt"
    torch.save(
        {
            "schema_version": RUN_CONFIG["schema_version"],
            "model_state_dict": FINAL_MODEL.state_dict(),
            "selected_config": SELECTED_CONFIG,
            "threshold_by_phase": THRESHOLD_BY_PHASE,
            "n_users": N_USERS,
            "n_items": N_ITEMS,
            "protocol_pointer_sha256": PROTOCOL_POINTER_SHA256,
        },
        checkpoint_path,
    )
    OUTPUT_ARTIFACTS["final_model_checkpoint"] = {
        "path": relative_output(generation_root / "final_model.pt"),
        "sha256": sha256_file(checkpoint_path),
    }

    manifest = {
        "model_schema_version": RUN_CONFIG["schema_version"],
        "model_status": "PASS",
        "model_name": "LightGCN",
        "bundle_id": bundle_id,
        "bundle_manifest": relative_output(generation_root / "manifest.json"),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "upstream_protocol": {
            "schema_version": PROTOCOL_MANIFEST["protocol_schema_version"],
            "bundle_id": PROTOCOL_MANIFEST["bundle_id"],
            "pointer_sha256": PROTOCOL_POINTER_SHA256,
        },
        "run_config": RUN_CONFIG,
        "run_config_sha256": RUN_CONFIG_HASH,
        "selected_config": selected_config_payload,
        "summary": {
            "selected_config_name": SELECTED_CONFIG_NAME,
            "mean_validation_f1": SELECTED_SUMMARY["mean_validation_f1"],
            "mean_validation_auc": SELECTED_SUMMARY["mean_validation_auc"],
            "mean_evaluation_f1": float(EVALUATION_METRICS["f1"].mean()),
            "mean_evaluation_auc": float(EVALUATION_METRICS["roc_auc"].mean()),
            "evaluation_prediction_rows": len(EVALUATION_PREDICTIONS),
            "final_training_seconds": float(FINAL_TRAINING_SECONDS),
        },
        "artifacts": OUTPUT_ARTIFACTS,
        "checks": PRE_EXPORT_CHECKS,
        "output_schemas": {name: csv_schema(table) for name, table in OUTPUT_TABLES.items()},
        "phase_contract": PROTOCOL_MANIFEST["phase_contract"],
        "training_contract": {
            "tuning_base": "tuning_train from notebook 02",
            "final_refit_base": "final_train from notebook 02",
            "graph_support": "Cold none; Warm A/B/C cumulative support rows",
            "features": "ID embeddings only; no side information",
            "thresholds": "selected on validation query rows only and frozen for evaluation",
        },
    }

    manifest_text = json.dumps(json_ready(manifest), indent=2, sort_keys=True) + "\n"
    write_text(staging_root / "manifest.json", manifest_text)
    generation_root.parent.mkdir(parents=True, exist_ok=True)
    staging_root.replace(generation_root)
    write_text(pointer_path, manifest_text)
except Exception:
    if staging_root.exists():
        shutil.rmtree(staging_root, ignore_errors=True)
    raise

LIGHTGCN_MANIFEST = manifest
LIGHTGCN_POINTER = pointer_path
display(EVALUATION_METRICS[["phase", "rows", "positives", "threshold", "f1", "roc_auc"]])
show_records(
    [
        {
            "model_status": "PASS",
            "bundle_id": bundle_id,
            "manifest": str(pointer_path),
            "artifacts": len(OUTPUT_ARTIFACTS),
        }
    ]
)

,phase,rows,positives,threshold,f1,roc_auc
0,Cold,57569,23501,-0.041051,0.579743,0.506046
1,Warm A,57569,23501,-0.037596,0.579732,0.503783
2,Warm B,57569,23501,-0.037102,0.579732,0.502547
3,Warm C,57569,23501,-0.037513,0.579732,0.502454


,model_status,bundle_id,manifest,artifacts
0,PASS,20260716T013724-7f5d845a2436,/kaggle/working/artifacts/models/ml-1m/lightgc...,10


## Checks and Continuation

Before moving on, require that every phase has predictions, thresholds came
from validation only, the model bundle is hash-verifiable, and new-item metrics
were produced under the shared protocol. Notebook 04 should consume the same
notebook-02 protocol bundle, not this LightGCN model bundle.

In [5]:
LIGHTGCN_CHECKS: list[dict[str, Any]] = list(PRE_EXPORT_CHECKS)


def check(name: str, condition: bool, observed: Any, expected: Any) -> None:
    LIGHTGCN_CHECKS.append(
        {"check": name, "status": "PASS" if condition else "FAIL", "observed": observed, "expected": expected}
    )


manifest_pointer_matches = LIGHTGCN_POINTER.read_text(encoding="utf-8") == (
    generation_root / "manifest.json"
).read_text(encoding="utf-8")
artifact_hashes_match = all(
    sha256_file(resolve_inside(ARTIFACT_ROOT, artifact["path"])) == artifact["sha256"]
    for artifact in LIGHTGCN_MANIFEST["artifacts"].values()
)
checkpoint_exported = "final_model_checkpoint" in LIGHTGCN_MANIFEST["artifacts"]

check("manifest pointer matches bundle", manifest_pointer_matches, manifest_pointer_matches, True)
check("artifact hashes verify", artifact_hashes_match, artifact_hashes_match, True)
check("checkpoint exported", checkpoint_exported, checkpoint_exported, True)

LIGHTGCN_PASS = all(row["status"] == "PASS" for row in LIGHTGCN_CHECKS)
display(pd.DataFrame(LIGHTGCN_CHECKS))
display(Markdown("### Notebook 03 LightGCN baseline: " + ("READY" if LIGHTGCN_PASS else "BLOCKED")))

if not LIGHTGCN_PASS:
    raise RuntimeError("LightGCN checks failed; inspect LIGHTGCN_CHECKS before continuing")

display(
    Markdown(
        "**Next notebook:** implement EmerG in notebook 04 with the same protocol manifest. "
        "Use notebook 03 only as the collaborative-filtering baseline result bundle for later comparison."
    )
)

,check,status,observed,expected
0,protocol schema,PASS,ml1m-coldstart-v1,ml1m-coldstart-v1
1,phase thresholds complete,PASS,"[Cold, Warm A, Warm B, Warm C]","[Cold, Warm A, Warm B, Warm C]"
2,validation predictions complete,PASS,611048,611048
3,evaluation predictions complete,PASS,230276,230276
4,evaluation phases complete,PASS,"[Cold, Warm A, Warm B, Warm C]","[Cold, Warm A, Warm B, Warm C]"
5,final train excludes evaluation items,PASS,True,True
6,manifest pointer matches bundle,PASS,True,True
7,artifact hashes verify,PASS,True,True
8,checkpoint exported,PASS,True,True


### Notebook 03 LightGCN baseline: READY

**Next notebook:** implement EmerG in notebook 04 with the same protocol manifest. Use notebook 03 only as the collaborative-filtering baseline result bundle for later comparison.